# GGUF 导出

GGUF 是 llama.cpp、Ollama、LM Studio 常用格式。它主要用于本地推理，不是训练格式。

如果模型来自 LoRA/QLoRA，通常先合并 adapter，再导出 GGUF。

## 1. 常见量化级别

- `Q4_K_M`：最常用平衡选择。体积小，质量尚可。
- `Q5_K_M`：质量更好，文件更大。
- `Q8_0`：更接近原始质量，占用更高。

用户口头说的 `qkkm` 通常应统一写成标准 GGUF 名称 `Q4_K_M`，避免混淆。

In [ ]:
merged_model_dir = "../../outputs/qwen2_5_7b/merged"
fp16_gguf = "../../outputs/qwen2_5_7b/model-f16.gguf"
q4_gguf = "../../outputs/qwen2_5_7b/model-q4_k_m.gguf"

convert_command = f"python convert_hf_to_gguf.py {merged_model_dir} --outfile {fp16_gguf} --outtype f16"
quantize_command = f"./llama-quantize {fp16_gguf} {q4_gguf} Q4_K_M"
run_command = f"./llama-cli -m {q4_gguf} -c 8192 -ngl 999 -p '解释 SFT 和 PEFT 的关系。'"

print(convert_command)
print(quantize_command)
print(run_command)

## 2. Ollama Modelfile

导出 GGUF 后，可以写 Ollama Modelfile 创建本地模型。

In [ ]:
modelfile = """FROM ./model-q4_k_m.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.7
PARAMETER top_p 0.9
SYSTEM 你是一个回答准确、表达清楚的中文助手。
"""

print(modelfile)
print("ollama create local-llm -f Modelfile")
print("ollama run local-llm")